# Original Qwen3-1.7B: unveränderte H286-Auswahl

**A100 bevorzugt, L4 möglich → Alle ausführen.** B200 ist bei passendem
Torch/CUDA und bestandenem Preflight ebenfalls möglich.
Dieselbe Datei funktioniert in Colab und in Jupyter. Eine CUDA-GPU mit mindestens
12 GiB freiem Speicher ist erforderlich; es gibt keine GPU-Warteschleife.

Die private Nutzdatei enthält die vorhandenen Referenzen bereits. Sie benötigt
keinen Drive-Zugriff und keine weiteren Uploads oder Pfadangaben. Das öffentliche
Template enthält keine privaten Referenzdaten und stoppt ohne sie vor Downloads.

Es werden die echten, festgelegten Qwen3-1.7B-Gewichte geladen (etwa 3,5 GB Download,
etwa 7 GB Modellgewichte in FP32 plus Laufzeitbedarf). Die 286 Koordinaten und ihre
originalen REF-W-Statistiken bleiben unverändert. Nach Warm-up und zwei technischen
Checks läuft der begrenzte Versuch mit 16 kurzen Fragen und fünf Bedingungen
automatisch. Fortschritt und Zeitabschätzung erscheinen während des Laufs.

Ergebnisse sind explorativ. Gemessene Rohaktivierung und geerbte Normierung werden
im Paket bezeichnet; dieser Versuch bestätigt keinen allgemeinen Wahrheits- oder
Selbstüberwachungsmechanismus. Atlas/Qwen3.6 bleibt separat vorbereitet.

Am Ende wird das Ergebnis- oder Diagnose-ZIP direkt angeboten. Ein bewusstes
Zeitlimit erzeugt einen fortsetzbaren Checkpoint mit Status `paused`, keinen PASS.
Nach dem Lauf die GPU-Sitzung beenden, wenn sie nicht mehr gebraucht wird.


In [ ]:
# Private Nutzdatei: unverändert lassen und Alle ausführen.
# Öffentliches Template: optional vorhandene Originaldateien in diesem Verzeichnis.
REFERENCE_DIRECTORY = ""
RESUME_ZIP = ""  # Nur für einen vorhandenen Checkpoint von einer anderen Sitzung.
RUN_DIRECTORY = ""  # Leer: automatisch, wiederholbar in derselben Sitzung.
MIN_FREE_VRAM_GIB = 12.0


In [ ]:
# Eingebettete Quellen und optionale private Originalbytes; nicht manuell ändern.
TRANSFORMERS_VERSION = '5.22.2'
ACCELERATE_VERSION = '1.16.0'
REFERENCE_SHA256 = {'hn_set.json': '11390233b62c63b9d34805482ecbd3675eadd80570b9c01ef9e268bf565c98fd', 'htq_policy.json': '160c420603b719afba4b909d12d31316535290e13b00aea2a6dea88110684f7c', 'refw_h286.json': '68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc'}
REFERENCE_FILES_B64 = {}
MODULE_SOURCES = {'attention_probe/__init__.py': '"""Isolated, optional attention-pulse pilot; existing experiment code is unchanged."""\n', 'attention_probe/reference.py': '"""Bind a new attention pilot to the private, exact DSSE H_tq selection.\n\nThis module validates identities and provenance only. It imports no model runtime,\ncontains no private neuron coordinates, and does not map them to another model.\n\n    python -m attention_probe.reference --hn-set /private/hn_set.json \\\n        --policy /private/htq_policy.json --out /private/reference.json\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom dataclasses import dataclass\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport tempfile\nfrom typing import Any\n\n\nHN_SET_SHA256 = "11390233b62c63b9d34805482ecbd3675eadd80570b9c01ef9e268bf565c98fd"\nPOLICY_SHA256 = "160c420603b719afba4b909d12d31316535290e13b00aea2a6dea88110684f7c"\nMODEL_ID = "qwen3-1.7b"\nMODEL_REPO = "Qwen/Qwen3-1.7B"\nMODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"\nN_COORDINATES, N_LAYERS, MLP_WIDTH = 286, 28, 6144\nH_POLICY = "htq_v2"\n\n\nclass ReferenceError(ValueError):\n    """The supplied files do not establish the required DSSE reference."""\n\n\n@dataclass(frozen=True, slots=True)\nclass ModelPins:\n    repo: str\n    revision: str\n    sha256: tuple[tuple[str, str], ...]\n    model_id: str = MODEL_ID\n    n_layers: int = N_LAYERS\n    mlp_width: int = MLP_WIDTH\n\n    def to_dict(self) -> dict[str, Any]:\n        return dict(model_id=self.model_id, repo=self.repo, revision=self.revision,\n                    sha256=dict(self.sha256), n_layers=self.n_layers, mlp_width=self.mlp_width)\n\n\n@dataclass(frozen=True, slots=True)\nclass Reference:\n    coordinates: tuple[tuple[int, int], ...]\n    hn_sha256: str\n    policy_sha256: str\n    model: ModelPins\n    h_policy: str = H_POLICY\n    post_hoc: bool = True\n    amendment: int = 2\n\n    def to_dict(self) -> dict[str, Any]:\n        """Return a fresh JSON binding; its coordinates remain private run data."""\n        return dict(schema="dsse_h286_reference_v1", coordinates=[list(x) for x in self.coordinates],\n                    coordinate_order="hn_set.H_op (identical to H_tq)",\n                    n_coordinates=len(self.coordinates),\n                    source_sha256={"hn_set.json": self.hn_sha256, "htq_policy.json": self.policy_sha256},\n                    model=self.model.to_dict(), h_policy=self.h_policy, post_hoc=self.post_hoc,\n                    amendment=self.amendment, confirmatory_eligible=False,\n                    scope="Neuron identities only; no CETT equivalence or cross-model coordinate mapping.")\n\n\ndef _digest(raw: bytes) -> str:\n    return hashlib.sha256(raw).hexdigest()\n\n\ndef _sha(value: Any, label: str) -> str:\n    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:\n        raise ReferenceError(f"{label} must be a lowercase SHA-256 digest.")\n    return value\n\n\ndef _object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:\n    out = {}\n    for key, value in pairs:\n        if key in out:\n            raise ReferenceError("Duplicate JSON object key.")\n        out[key] = value\n    return out\n\n\ndef _reject_constant(value: str) -> Any:\n    raise ReferenceError("Non-finite JSON constants are not allowed.")\n\n\ndef _document(raw: bytes, label: str) -> dict[str, Any]:\n    try:\n        value = json.loads(raw, object_pairs_hook=_object, parse_constant=_reject_constant)\n    except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise ReferenceError(f"{label} is not valid JSON.") from exc\n    if not isinstance(value, dict):\n        raise ReferenceError(f"{label} must be a JSON object.")\n    return value\n\n\ndef _coordinates(value: Any, label: str) -> tuple[tuple[int, int], ...]:\n    if not isinstance(value, list) or len(value) != N_COORDINATES:\n        raise ReferenceError(f"{label} must contain exactly {N_COORDINATES} coordinates.")\n    result = []\n    for i, coordinate in enumerate(value):\n        if not isinstance(coordinate, list) or len(coordinate) != 2 or any(type(x) is not int for x in coordinate):\n            raise ReferenceError(f"{label}[{i}] must be a pair of integer coordinates.")\n        layer, index = coordinate\n        if not (0 <= layer < N_LAYERS and 0 <= index < MLP_WIDTH):\n            raise ReferenceError(f"{label}[{i}] is outside the 28 by 6144 MLP architecture.")\n        result.append((layer, index))\n    if len(set(result)) != N_COORDINATES:\n        raise ReferenceError(f"{label} contains duplicate coordinates.")\n    return tuple(result)\n\n\ndef _policy(document: dict[str, Any], label: str) -> None:\n    if document.get("h_policy") != H_POLICY or document.get("post_hoc") is not True:\n        raise ReferenceError(f"{label} must identify the post-hoc htq_v2 policy.")\n    if type(document.get("amendment")) is not int or document["amendment"] != 2:\n        raise ReferenceError(f"{label} must identify amendment 2.")\n    if document.get("H_op_name") != "H_tq":\n        raise ReferenceError(f"{label} must use H_tq as the operative set.")\n\n\ndef _model(document: dict[str, Any], label: str) -> ModelPins:\n    provenance = document.get("provenance")\n    if not isinstance(provenance, dict) or not isinstance(provenance.get("models"), dict):\n        raise ReferenceError(f"{label} has no model provenance.")\n    for field, expected in (("h_policy", H_POLICY), ("post_hoc", True), ("amendment", 2), ("mock", False)):\n        if field in provenance and (type(provenance[field]) is not type(expected) or provenance[field] != expected):\n            raise ReferenceError(f"{label} has conflicting {field} provenance.")\n    model = provenance["models"].get(MODEL_ID)\n    if not isinstance(model, dict) or model.get("repo") != MODEL_REPO or model.get("revision") != MODEL_REVISION:\n        raise ReferenceError(f"{label} does not identify the pinned Qwen3-1.7B model.")\n    hashes = model.get("sha256")\n    if not isinstance(hashes, dict) or not hashes or any(not isinstance(k, str) or not k for k in hashes):\n        raise ReferenceError(f"{label} has no model-file SHA-256 pins.")\n    pins = tuple(sorted((name, _sha(digest, f"{label} model file")) for name, digest in hashes.items()))\n    return ModelPins(repo=model["repo"], revision=model["revision"], sha256=pins)\n\n\ndef verify_documents(hn_bytes: bytes, policy_bytes: bytes, *, expected_hn_sha256: str,\n                     expected_policy_sha256: str) -> Reference:\n    """Low-level verification with explicit trust anchors, including synthetic tests.\n\n    Normal pilot code must call load_reference(), whose anchors are the production\n    hashes. Checksums cover original bytes, so JSON reformatting is not equivalent.\n    """\n    expected_hn_sha256 = _sha(expected_hn_sha256, "Expected hn_set hash")\n    expected_policy_sha256 = _sha(expected_policy_sha256, "Expected policy hash")\n    if not isinstance(hn_bytes, bytes) or not isinstance(policy_bytes, bytes):\n        raise ReferenceError("Reference documents must be immutable bytes.")\n    hn_digest, policy_digest = _digest(hn_bytes), _digest(policy_bytes)\n    if hn_digest != expected_hn_sha256:\n        raise ReferenceError("hn_set.json SHA-256 mismatch.")\n    if policy_digest != expected_policy_sha256:\n        raise ReferenceError("htq_policy.json SHA-256 mismatch.")\n    hn, policy = _document(hn_bytes, "hn_set.json"), _document(policy_bytes, "htq_policy.json")\n    _policy(hn, "hn_set.json"); _policy(policy, "htq_policy.json")\n    coordinates = _coordinates(hn.get("H_op"), "H_op")\n    if coordinates != _coordinates(hn.get("H_tq"), "H_tq"):\n        raise ReferenceError("H_op and H_tq must be identical, including coordinate order.")\n    if type(hn.get("n_H")) is not int or hn["n_H"] != N_COORDINATES:\n        raise ReferenceError("hn_set n_H must equal 286.")\n    if type(policy.get("n_H_op")) is not int or policy["n_H_op"] != N_COORDINATES:\n        raise ReferenceError("Policy n_H_op must equal 286.")\n    hn_model, policy_model = _model(hn, "hn_set.json"), _model(policy, "htq_policy.json")\n    if hn_model != policy_model:\n        raise ReferenceError("The two files have different model-file pins.")\n    return Reference(coordinates=coordinates, hn_sha256=hn_digest,\n                     policy_sha256=policy_digest, model=hn_model)\n\n\ndef load_reference(hn_set_path: str | Path, policy_path: str | Path) -> Reference:\n    """Load only the exact DSSE production files; no caller-supplied hash bypass."""\n    return verify_documents(Path(hn_set_path).read_bytes(), Path(policy_path).read_bytes(),\n                            expected_hn_sha256=HN_SET_SHA256, expected_policy_sha256=POLICY_SHA256)\n\n\ndef write_reference(reference: Reference, out: str | Path) -> None:\n    """Atomically write a binding to the caller\'s explicit private output path."""\n    path = Path(out)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = json.dumps(reference.to_dict(), indent=2, ensure_ascii=False, allow_nan=False) + "\\n"\n    temporary = None\n    try:\n        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,\n                                         prefix="." + path.name + ".", suffix=".tmp", delete=False) as handle:\n            temporary = Path(handle.name)\n            handle.write(payload); handle.flush(); os.fsync(handle.fileno())\n        os.replace(temporary, path)\n    finally:\n        if temporary is not None:\n            temporary.unlink(missing_ok=True)\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--hn-set", required=True, type=Path)\n    parser.add_argument("--policy", required=True, type=Path)\n    parser.add_argument("--out", required=True, type=Path)\n    args = parser.parse_args(argv)\n    if args.out.resolve() in (args.hn_set.resolve(), args.policy.resolve()):\n        parser.error("--out must not overwrite a source document.")\n    try:\n        reference = load_reference(args.hn_set, args.policy)\n        write_reference(reference, args.out)\n    except (OSError, ReferenceError) as exc:\n        parser.error(str(exc))\n    print(f"Validated {len(reference.coordinates)} H_tq coordinates; private binding written to {args.out}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'attention_probe/fixed_refw.py': '"""Validate the private, fixed H286 REF-W extract without model imports.\n\nThe compact file contains existing population means/SDs, already cast to FP32\nas in DSSE\'s SetReader. This module neither fits nor recomputes normalization.\nCETT uses max(||down_proj(z)||, CETT_EPS); zCETT uses the saved c_mu and\nmax(c_sd, STANDARDIZATION_EPS). Zero saved SD is valid and stays zero here:\nthe consuming measurement applies the documented denominator floor.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nimport struct\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .reference import (\n    HN_SET_SHA256,\n    MLP_WIDTH,\n    N_COORDINATES,\n    N_LAYERS,\n    POLICY_SHA256,\n    Reference,\n    ReferenceError,\n)\n\nFIXED_REFW_SHA256 = "68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc"\nSOURCE_HN_META_SHA256 = "9c61bc34e804719e5bf5e72b644e8a38a94ad0f6c1708efaef75386acdae49ff"\nCETT_EPS = 1e-9\nSTANDARDIZATION_EPS = 1e-6\nSCHEMA = "dsse_h286_refw_v1"\n_VECTOR_NAMES = {"z_mu", "z_sd", "c_mu", "c_sd"}\n_FIELDS = {"schema", "source_hn_meta_sha256", "hn_sha256", "coordinates", "refw"}\n\n\nclass FixedRefWError(ReferenceError):\n    """The supplied extract does not establish the fixed normalization."""\n\n\ndef _sha(value: Any, label: str) -> str:\n    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:\n        raise FixedRefWError(f"{label} must be a lowercase SHA-256 digest.")\n    return value\n\n\ndef _object(pairs: list[tuple[str, Any]]) -> dict[str, Any]:\n    result = {}\n    for key, value in pairs:\n        if key in result:\n            raise FixedRefWError("Duplicate JSON object key.")\n        result[key] = value\n    return result\n\n\ndef _reject_constant(value: str) -> Any:\n    raise FixedRefWError(f"Non-finite JSON constant: {value}")\n\n\ndef verify_fixed_refw(\n    raw: bytes,\n    reference: Reference,\n    *,\n    expected_sha256: str,\n    expected_source_hn_meta_sha256: str,\n    expected_hn_sha256: str,\n) -> dict[str, Any]:\n    """Low-level validation with explicit trust anchors for synthetic tests.\n\n    Production callers must use load_fixed_refw(), which fixes all three\n    anchors. The source hash binds the previously verified full hn_meta file;\n    this compact loader does not claim to inspect that large file again.\n    """\n    expected_sha256 = _sha(expected_sha256, "Expected compact hash")\n    expected_source_hn_meta_sha256 = _sha(expected_source_hn_meta_sha256, "Expected source hash")\n    expected_hn_sha256 = _sha(expected_hn_sha256, "Expected H selection hash")\n    if not isinstance(raw, bytes):\n        raise FixedRefWError("The compact document must be immutable bytes.")\n    if hashlib.sha256(raw).hexdigest() != expected_sha256:\n        raise FixedRefWError("Fixed REF-W SHA-256 mismatch.")\n    if not isinstance(reference, Reference) or reference.hn_sha256 != expected_hn_sha256:\n        raise FixedRefWError("Reference has a different H selection hash.")\n    try:\n        document = json.loads(raw, object_pairs_hook=_object, parse_constant=_reject_constant)\n    except (UnicodeDecodeError, json.JSONDecodeError) as exc:\n        raise FixedRefWError("Fixed REF-W is not valid JSON.") from exc\n    if not isinstance(document, dict) or set(document) != _FIELDS:\n        raise FixedRefWError("Unexpected fixed REF-W document fields.")\n    if document["schema"] != SCHEMA:\n        raise FixedRefWError("Unsupported fixed REF-W schema.")\n    if document["source_hn_meta_sha256"] != expected_source_hn_meta_sha256:\n        raise FixedRefWError("Full hn_meta source hash mismatch.")\n    if document["hn_sha256"] != expected_hn_sha256:\n        raise FixedRefWError("Compact H selection hash mismatch.")\n\n    coordinates = document["coordinates"]\n    if not isinstance(coordinates, list) or len(coordinates) != N_COORDINATES:\n        raise FixedRefWError("Expected exactly 286 ordered coordinates.")\n    checked = []\n    for coordinate in coordinates:\n        if (not isinstance(coordinate, list) or len(coordinate) != 2\n                or any(type(value) is not int for value in coordinate)):\n            raise FixedRefWError("Coordinates must be integer pairs.")\n        layer, neuron = coordinate\n        if not (0 <= layer < N_LAYERS and 0 <= neuron < MLP_WIDTH):\n            raise FixedRefWError("Coordinate outside the Qwen3-1.7B MLP architecture.")\n        checked.append((layer, neuron))\n    if len(set(checked)) != N_COORDINATES or tuple(checked) != reference.coordinates:\n        raise FixedRefWError("Coordinates differ from the ordered H286 reference.")\n\n    vectors = document["refw"]\n    if not isinstance(vectors, dict) or set(vectors) != _VECTOR_NAMES:\n        raise FixedRefWError("Expected exactly z_mu, z_sd, c_mu and c_sd.")\n    for name, vector in vectors.items():\n        if not isinstance(vector, list) or len(vector) != N_COORDINATES:\n            raise FixedRefWError(f"{name} must have exactly 286 values.")\n        for value in vector:\n            if type(value) not in (int, float):\n                raise FixedRefWError(f"{name} must contain finite FP32 numbers.")\n            try:\n                fp32 = struct.unpack("<f", struct.pack("<f", value))[0]\n            except (OverflowError, struct.error) as exc:\n                raise FixedRefWError(f"{name} contains a value outside FP32.") from exc\n            if not math.isfinite(fp32) or fp32 != value:\n                raise FixedRefWError(f"{name} contains a non-finite or non-FP32 value.")\n            if name.endswith("_sd") and value < 0:\n                raise FixedRefWError(f"{name} cannot contain negative standard deviations.")\n    return document\n\n\ndef load_fixed_refw(path: str | Path, reference: Reference) -> dict[str, Any]:\n    """Load only the exact production extract bound to the original reference."""\n    if not isinstance(reference, Reference) or reference.policy_sha256 != POLICY_SHA256:\n        raise FixedRefWError("Expected the original production reference policy.")\n    return verify_fixed_refw(\n        Path(path).read_bytes(), reference,\n        expected_sha256=FIXED_REFW_SHA256,\n        expected_source_hn_meta_sha256=SOURCE_HN_META_SHA256,\n        expected_hn_sha256=HN_SET_SHA256,\n    )\n', 'attention_probe/interventions.py': '"""One-pass attention pulses and read-only activation capture for Qwen3.\n\nPositions are zero-based columns of the complete input sequence, not relative\nindices of a cached decode chunk. Selected positions must be real tokens in\nevery batch row. Qwen3 queries are scaled *after* q_norm and before RoPE, so\nnormalization cannot cancel the pulse. Scaling queries scales their attention\nlogits; it makes no claim about FLOPs, energy, or a computational bottleneck.\n\nCoordinateCapture stores raw gated-MLP z at down_proj input, not CETT.\n"""\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nimport math\nimport inspect\nfrom numbers import Integral\nfrom typing import Iterable, Sequence\n\nimport torch\n\n\ndef _integers(values: Iterable[int], name: str, *, nonempty: bool = True) -> tuple[int, ...]:\n    result = tuple(values)\n    if nonempty and not result:\n        raise ValueError(f"{name} must not be empty")\n    if any(isinstance(x, bool) or not isinstance(x, Integral) or x < 0 for x in result):\n        raise ValueError(f"{name} must contain nonnegative integers")\n    if len(set(result)) != len(result):\n        raise ValueError(f"{name} must not contain duplicates")\n    return tuple(int(x) for x in result)\n\n\ndef _layers(model):\n    base = getattr(model, "model", model)\n    layers = getattr(base, "layers", None)\n    if layers is None or not len(layers):\n        raise ValueError("Expected Hugging Face decoder model.model.layers (or model.layers)")\n    return layers\n\n\ndef _cache_length(cache) -> int | None:\n    if cache is None:\n        return 0\n    getter = getattr(cache, "get_seq_length", None)\n    if getter is not None:\n        try:\n            return int(getter())\n        except (TypeError, ValueError, RuntimeError):\n            return None\n    if isinstance(cache, (tuple, list)):\n        if not cache:\n            return 0\n        try:\n            return int(cache[0][0].shape[-2])\n        except (IndexError, AttributeError, TypeError):\n            return None\n    return None\n\n\nclass _OneFullPass:\n    """Bind hooks to one complete, uncached outer model forward and clean them up."""\n\n    def __init__(self, model, positions: Sequence[int]):\n        self.model = model\n        self.positions = _integers(positions, "positions")\n        self._handles = []\n        self._entered = False\n        self._used = False\n        self._claimed = False\n        self._window = None\n        self.full_passes = 0\n        self.skipped_cached_passes = 0\n\n    def _model_pre(self, module, args, kwargs):\n        self._window = None\n        if self._claimed:\n            return\n        # HF forwards allow both positional and keyword cache arguments.\n        try:\n            named = inspect.signature(module.forward).bind_partial(*args, **kwargs).arguments\n        except (TypeError, ValueError):\n            named = kwargs\n        fields = dict(kwargs)\n        fields.update({name: value for name, value in named.items() if name not in ("args", "kwargs")})\n        cache = fields.get("past_key_values", fields.get("past_key_value"))\n        length = _cache_length(cache)\n        cache_position = fields.get("cache_position")\n        nonzero_offset = (cache_position is not None and torch.as_tensor(cache_position).numel()\n                          and int(torch.as_tensor(cache_position).reshape(-1)[0]) != 0)\n        if length is None or length > 0 or nonzero_offset:\n            self.skipped_cached_passes += 1\n            return\n        ids = fields.get("input_ids")\n        embeds = fields.get("inputs_embeds")\n        value = ids if ids is not None else embeds\n        if value is None and args:\n            value = args[0]\n        if not isinstance(value, torch.Tensor) or value.ndim not in (2, 3):\n            raise ValueError("Cannot identify full input sequence for absolute token positions")\n        batch, tokens = value.shape[:2]\n        if max(self.positions) >= tokens:\n            raise ValueError(f"position {max(self.positions)} is outside the full sequence of {tokens} tokens")\n        mask = fields.get("attention_mask")\n        if mask is not None:\n            if tuple(mask.shape) != (batch, tokens):\n                raise ValueError("Expected a two-dimensional outer attention mask")\n            if not bool(mask[:, list(self.positions)].bool().all()):\n                raise ValueError("Selected absolute position is padding in at least one batch row")\n        self._claimed = True\n        self.full_passes += 1\n        self._window = (int(batch), int(tokens))\n\n    def _model_post(self, module, args, kwargs, output):\n        self._window = None\n\n    def _install(self):\n        raise NotImplementedError\n\n    def validate_fired(self):\n        raise NotImplementedError\n\n    def __enter__(self):\n        if self._used:\n            raise RuntimeError("Use a fresh hook context for each attempt")\n        if self.model.training:\n            raise ValueError("Attention pilot requires model.eval(); model mode is never changed by hooks")\n        self._used = self._entered = True\n        try:\n            self._handles.append(self.model.register_forward_pre_hook(self._model_pre, with_kwargs=True))\n            self._handles.append(self.model.register_forward_hook(self._model_post, with_kwargs=True, always_call=True))\n            self._install()\n        except BaseException:\n            self.close()\n            raise\n        return self\n\n    def close(self):\n        for handle in reversed(self._handles):\n            handle.remove()\n        self._handles.clear()\n        self._window = None\n        self._entered = False\n\n    def __exit__(self, kind, error, tb):\n        self.close()\n        if kind is None:\n            self.validate_fired()\n        return False\n\n\nclass AttentionPulse(_OneFullPass):\n    """Scale post-q_norm queries, or add a per-head matched random delta, once.\n\n    Primary architecture is config.model_type=\'qwen3\'. Qwen2\'s unnormalized\n    q_proj output is supported only with explicit architecture=\'qwen2\'. Random\n    deltas match ||(beta-1) q|| independently for each selected token/query head\n    before output-dtype rounding. actual_delta_norms reports rounding effects.\n    A local CPU generator never reads or changes PyTorch\'s global RNG state.\n    """\n\n    def __init__(self, model, block: int, positions: Sequence[int], beta: float = 1.0,\n                 mode: str = "scale", seed: int = 0, *, architecture: str = "qwen3"):\n        super().__init__(model, positions)\n        if not isinstance(beta, (int, float)) or not math.isfinite(beta) or not 0 <= beta <= 1:\n            raise ValueError("beta must be finite and between zero and one")\n        if mode not in ("scale", "random"):\n            raise ValueError("mode must be \'scale\' or \'random\'")\n        if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:\n            raise ValueError("seed must be a nonnegative integer")\n        self.layers = _layers(model)\n        self.block = _integers((block,), "block")[0]\n        if self.block >= len(self.layers):\n            raise ValueError("block is outside the decoder")\n        if architecture not in ("qwen3", "qwen2") or getattr(model.config, "model_type", None) != architecture:\n            raise ValueError("Unsupported architecture: Qwen3 is primary; Qwen2 requires explicit opt-in")\n        self.architecture, self.mode, self.beta = architecture, mode, float(beta)\n        self.seed = int(seed)\n        attention = self.layers[self.block].self_attn\n        self.heads = int(model.config.num_attention_heads)\n        self.head_dim = int(getattr(attention, "head_dim", getattr(model.config, "head_dim", 0)))\n        if self.heads <= 0 or self.head_dim <= 0:\n            raise ValueError("Query-head shape is not declared")\n        if architecture == "qwen3":\n            self.hook_module = getattr(attention, "q_norm", None)\n            self.hook_location = "self_attn.q_norm.output (B,T,H,D), before RoPE"\n            if self.hook_module is None:\n                raise ValueError("Qwen3 requires post-q_norm intervention; q_proj fallback is forbidden")\n        else:\n            if getattr(attention, "q_norm", None) is not None:\n                raise ValueError("Qwen2 fallback expects no query normalization")\n            self.hook_module = getattr(attention, "q_proj", None)\n            self.hook_location = "self_attn.q_proj.output, explicit Qwen2 fallback"\n            if self.hook_module is None:\n                raise ValueError("Qwen2 query projection is absent")\n        self.fired_count = 0\n        self.requested_delta_norms = None\n        self.actual_delta_norms = None\n        self._generator = torch.Generator(device="cpu").manual_seed(self.seed)\n\n    def _install(self):\n        self._handles.append(self.hook_module.register_forward_hook(self._pulse))\n\n    def _pulse(self, module, args, output):\n        if self._window is None or self.fired_count:\n            return None\n        batch, tokens = self._window\n        if not isinstance(output, torch.Tensor):\n            raise ValueError("Query hook did not return a tensor")\n        shape = (batch, tokens, self.heads, self.head_dim)\n        if self.architecture == "qwen3":\n            if tuple(output.shape) != shape:\n                raise ValueError(f"Unsupported Qwen3 post-q_norm layout: expected {shape}, got {tuple(output.shape)}")\n            query = output\n        else:\n            if tuple(output.shape) != (batch, tokens, self.heads * self.head_dim):\n                raise ValueError("Unsupported Qwen2 q_proj output shape")\n            query = output.reshape(shape)\n        selected = query[:, list(self.positions), :, :]\n        if not bool(torch.isfinite(selected).all()):\n            raise ValueError("Nonfinite query in selected pulse positions")\n        self.fired_count += 1\n        norms = (1-self.beta) * selected.detach().double().cpu().norm(dim=-1)\n        self.requested_delta_norms = norms.clone()\n        if self.beta == 1:\n            self.actual_delta_norms = torch.zeros_like(norms)\n            return None  # Exact sham: preserve the original tensor, no multiplication or RNG draw.\n        if self.mode == "scale":\n            changed = selected * self.beta\n        else:\n            random = torch.randn(selected.shape, generator=self._generator, dtype=torch.float64, device="cpu")\n            random = random / random.norm(dim=-1, keepdim=True).clamp_min(torch.finfo(torch.float64).tiny)\n            delta = (random * norms[..., None]).to(device=selected.device, dtype=selected.dtype)\n            changed = selected + delta\n        self.actual_delta_norms = (changed.detach().double().cpu()-selected.detach().double().cpu()).norm(dim=-1)\n        result = query.clone()\n        result[:, list(self.positions), :, :] = changed\n        return result if self.architecture == "qwen3" else result.reshape(output.shape)\n\n    def validate_fired(self):\n        if self.fired_count != 1:\n            raise RuntimeError(f"Attention pulse must fire once on a full/prefill pass; fired {self.fired_count}")\n        return self.fired_count\n\n\nclass CoordinateCapture(_OneFullPass):\n    """Read exact ordered down_proj-input coordinates and decoder residuals.\n\n    values is CPU float32 [batch, positions, coordinates]; coordinate order is\n    exactly the supplied order. residuals maps each requested decoder layer to\n    CPU float32 [batch, positions, hidden_size]. No input/output tensor changes.\n    This is raw z capture, not CETT or a causal-neuron importance estimate.\n    """\n\n    feature_kind = "raw gated-MLP activation z at down_proj input; not CETT"\n\n    def __init__(self, model, coordinates: Sequence[tuple[int, int]], positions: Sequence[int],\n                 residual_layers: Sequence[int] | None = None, *, expected_count: int = 286):\n        super().__init__(model, positions)\n        self.layers = _layers(model)\n        if getattr(model.config, "model_type", None) != "qwen3":\n            raise ValueError("Original H coordinates require the Qwen3 decoder architecture")\n        if isinstance(expected_count, bool) or not isinstance(expected_count, Integral) or expected_count < 1:\n            raise ValueError("expected_count must be a positive integer")\n        self.coordinates = tuple(tuple(pair) for pair in coordinates)\n        if len(self.coordinates) != expected_count or not self.coordinates:\n            raise ValueError(f"Expected exactly {expected_count} ordered H coordinates")\n        if len(set(self.coordinates)) != len(self.coordinates):\n            raise ValueError("Duplicate H coordinates")\n        self._by_layer = defaultdict(list)\n        for output_index, pair in enumerate(self.coordinates):\n            if len(pair) != 2:\n                raise ValueError("Each coordinate must be (block, down_proj_input_index)")\n            # A coordinate may have equal block/channel numbers; duplicates are\n            # forbidden across coordinate pairs, not within a pair.\n            if any(isinstance(x, bool) or not isinstance(x, Integral) or x < 0 for x in pair):\n                raise ValueError("Coordinate indices must be nonnegative integers")\n            block, neuron = (int(x) for x in pair)\n            if block >= len(self.layers):\n                raise ValueError("Coordinate block is outside the decoder")\n            down = getattr(getattr(self.layers[block], "mlp", None), "down_proj", None)\n            if down is None or not hasattr(down, "in_features") or neuron >= down.in_features:\n                raise ValueError("Coordinate channel is outside mlp.down_proj input")\n            self._by_layer[block].append((output_index, neuron))\n        self.residual_layers = _integers(residual_layers if residual_layers is not None else sorted(self._by_layer),\n                                        "residual_layers", nonempty=False)\n        if any(block >= len(self.layers) for block in self.residual_layers):\n            raise ValueError("Residual layer is outside the decoder")\n        self._values = None\n        self.residuals = {}\n        self._seen = set()\n        self.fired_count = 0\n\n    def _install(self):\n        for block in self._by_layer:\n            self._handles.append(self.layers[block].mlp.down_proj.register_forward_pre_hook(self._coordinate_hook(block)))\n        for block in self.residual_layers:\n            self._handles.append(self.layers[block].register_forward_hook(self._residual_hook(block)))\n\n    def _coordinate_hook(self, block):\n        def hook(module, args):\n            if self._window is None:\n                return None\n            if block in self._seen:\n                raise RuntimeError("down_proj was called twice in the selected full pass")\n            z = args[0]\n            batch, tokens = self._window\n            if not isinstance(z, torch.Tensor) or z.ndim != 3 or tuple(z.shape[:2]) != (batch, tokens):\n                raise ValueError("Unsupported down_proj input shape")\n            pairs = self._by_layer[block]\n            observed = z[:, list(self.positions), :][:, :, [channel for _, channel in pairs]].detach().float().cpu().clone()\n            if not bool(torch.isfinite(observed).all()):\n                raise ValueError("Nonfinite selected raw activation")\n            if self._values is None:\n                self._values = torch.empty((batch, len(self.positions), len(self.coordinates)), dtype=torch.float32)\n            self._values[:, :, [index for index, _ in pairs]] = observed\n            self._seen.add(block)\n            self.fired_count = int(self._seen == set(self._by_layer))\n            return None\n        return hook\n\n    def _residual_hook(self, block):\n        def hook(module, args, output):\n            if self._window is None:\n                return None\n            if block in self.residuals:\n                raise RuntimeError("Decoder layer was called twice in the selected full pass")\n            value = output[0] if isinstance(output, (tuple, list)) else output\n            if not isinstance(value, torch.Tensor) or value.ndim != 3 or tuple(value.shape[:2]) != self._window:\n                raise ValueError("Unsupported decoder residual output shape")\n            observed = value[:, list(self.positions)].detach().float().cpu().clone()\n            if not bool(torch.isfinite(observed).all()):\n                raise ValueError("Nonfinite selected decoder residual")\n            self.residuals[block] = observed\n            return None\n        return hook\n\n    @property\n    def values(self):\n        self.validate_fired()\n        return self._values\n\n    def validate_fired(self):\n        if self.fired_count != 1 or self._seen != set(self._by_layer) or set(self.residuals) != set(self.residual_layers):\n            raise RuntimeError("Raw-coordinate capture did not complete exactly one full/prefill pass")\n        return self.fired_count\n', 'attention_probe/pilot_items.json': '[\n  {\n    "id": "p01_correct_draft",\n    "pair_id": "p01",\n    "prompt": "7 + 8 = ?. Draft: 15. Give only the correct number.",\n    "target": "15",\n    "category": "arithmetic_add"\n  },\n  {\n    "id": "p01_wrong_draft",\n    "pair_id": "p01",\n    "prompt": "7 + 8 = ?. Draft: 14. Give only the correct number.",\n    "target": "15",\n    "category": "arithmetic_add"\n  },\n  {\n    "id": "p02_correct_draft",\n    "pair_id": "p02",\n    "prompt": "9 - 4 = ?. Draft: 5. Give only the correct number.",\n    "target": "5",\n    "category": "arithmetic_subtract"\n  },\n  {\n    "id": "p02_wrong_draft",\n    "pair_id": "p02",\n    "prompt": "9 - 4 = ?. Draft: 6. Give only the correct number.",\n    "target": "5",\n    "category": "arithmetic_subtract"\n  },\n  {\n    "id": "p03_correct_draft",\n    "pair_id": "p03",\n    "prompt": "6 * 3 = ?. Draft: 18. Give only the correct number.",\n    "target": "18",\n    "category": "arithmetic_multiply"\n  },\n  {\n    "id": "p03_wrong_draft",\n    "pair_id": "p03",\n    "prompt": "6 * 3 = ?. Draft: 19. Give only the correct number.",\n    "target": "18",\n    "category": "arithmetic_multiply"\n  },\n  {\n    "id": "p04_correct_draft",\n    "pair_id": "p04",\n    "prompt": "20 / 5 = ?. Draft: 4. Give only the correct number.",\n    "target": "4",\n    "category": "arithmetic_divide"\n  },\n  {\n    "id": "p04_wrong_draft",\n    "pair_id": "p04",\n    "prompt": "20 / 5 = ?. Draft: 3. Give only the correct number.",\n    "target": "4",\n    "category": "arithmetic_divide"\n  },\n  {\n    "id": "p05_correct_draft",\n    "pair_id": "p05",\n    "prompt": "The key is in the blue box. Draft: blue. Give the box color only.",\n    "target": "blue",\n    "category": "context_fact"\n  },\n  {\n    "id": "p05_wrong_draft",\n    "pair_id": "p05",\n    "prompt": "The key is in the blue box. Draft: pink. Give the box color only.",\n    "target": "blue",\n    "category": "context_fact"\n  },\n  {\n    "id": "p06_correct_draft",\n    "pair_id": "p06",\n    "prompt": "The card was blue; now it is pink. Draft: pink. Give its current color only.",\n    "target": "pink",\n    "category": "context_update"\n  },\n  {\n    "id": "p06_wrong_draft",\n    "pair_id": "p06",\n    "prompt": "The card was blue; now it is pink. Draft: blue. Give its current color only.",\n    "target": "pink",\n    "category": "context_update"\n  },\n  {\n    "id": "p07_correct_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Draft: green. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application"\n  },\n  {\n    "id": "p07_wrong_draft",\n    "pair_id": "p07",\n    "prompt": "Every wug is green. Pip is a wug. Draft: brown. Give Pip\'s color only.",\n    "target": "green",\n    "category": "rule_application"\n  },\n  {\n    "id": "p08_correct_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Draft: B. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic"\n  },\n  {\n    "id": "p08_wrong_draft",\n    "pair_id": "p08",\n    "prompt": "Exactly one of A and B is on. A is off. Draft: A. Which is on? Reply A or B.",\n    "target": "B",\n    "category": "exclusive_logic"\n  }\n]\n', 'qwen3_h286_runtime.py': '"""Short exploratory ORIGINAL Qwen3/H286 pilot; embedded notebook entry point.\n\nNo re-selection, fitting, generation, training, or cross-model coordinate mapping.\nOriginal FP32, TF32-off, fixed REF-W and efficient SDPA are retained. The two\ntechnical checks precede a fixed 16-request/five-arm exploratory run. Results do\nnot constitute a held-out confirmatory test or a proven attention bottleneck.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport datetime as dt\nimport hashlib\nimport json\nimport os\nimport platform\nimport random\nimport shutil\nimport tempfile\nimport time\nimport traceback\nimport zipfile\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nMODEL = \'Qwen/Qwen3-1.7B\'\nREVISION = \'70d244cc86ccca08cf5af4e1e306ecf908b1ad5e\'\nTRANSFORMERS = \'5.22.2\'\nACCELERATE = \'1.16.0\'\nREFW_SHA256 = \'68f96129d8ab00f3c7e6a68a9b4a0bdb58798dfac4d1d405f6cce5ccca0ae5cc\'\nMETA_SHA256 = \'9c61bc34e804719e5bf5e72b644e8a38a94ad0f6c1708efaef75386acdae49ff\'\nSEED = 20261117\nARMS = (\n    {\'name\': \'sham\', \'block\': 14, \'beta\': 1., \'mode\': \'scale\'},\n    {\'name\': \'soft_103\', \'block\': 14, \'beta\': 1 / 1.03, \'mode\': \'scale\'},\n    {\'name\': \'soft_108\', \'block\': 14, \'beta\': 1 / 1.08, \'mode\': \'scale\'},\n    {\'name\': \'random_matched_108\', \'block\': 14, \'beta\': 1 / 1.08, \'mode\': \'random\'},\n    {\'name\': \'other_block_108\', \'block\': 7, \'beta\': 1 / 1.08, \'mode\': \'scale\'},\n)\n# Declared before any new model result. These are technical/exploratory items,\n# not the frozen DSSE evaluation sample or a benchmark accuracy test.\nITEMS_SHA256 = \'12a3cba862185b9f25ca39ea3ea49bd3bb6fc5f248c12dbbe02bcc8836663fb3\'\n\n\ndef canonical(value):\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), ensure_ascii=False, allow_nan=False).encode()\n\n\ndef sha_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 << 20), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.NamedTemporaryFile(dir=path.parent, prefix=\'.\' + path.name, delete=False) as handle:\n        temporary = Path(handle.name)\n        handle.write(canonical(value) + b\'\\n\')\n        handle.flush()\n        os.fsync(handle.fileno())\n    try:\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef archive(output):\n    target = output / \'qwen3_h286_results.zip\'\n    temporary = output / \'.qwen3_h286_results.zip.tmp\'\n    with zipfile.ZipFile(temporary, \'w\', zipfile.ZIP_DEFLATED) as bundle:\n        for item in sorted(output.rglob(\'*\')):\n            if item.is_file() and item not in (target, temporary) and not any(part.startswith(\'.\') for part in item.relative_to(output).parts):\n                bundle.write(item, item.relative_to(output))\n    os.replace(temporary, target)\n    return target\n\n\ndef load_inputs(hn_path, policy_path, refw_path, items_path):\n    from attention_probe.fixed_refw import load_fixed_refw\n    from attention_probe.reference import load_reference\n    reference = load_reference(hn_path, policy_path)\n    stats = load_fixed_refw(refw_path, reference)\n    raw = Path(items_path).read_bytes()\n    if hashlib.sha256(raw).hexdigest() != ITEMS_SHA256:\n        raise ValueError(\'The exact prespecified 16-item file is required; no replacement or post-result selection\')\n    items = json.loads(raw)\n    if len(items) != 16 or len({item[\'id\'] for item in items}) != 16:\n        raise ValueError(\'Exactly 16 unique fixed requests are required\')\n    return reference, stats, items\n\n\ndef hardware_preflight(torch, cache_dir):\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA GPU erforderlich; keine Warteschleife. A100 empfohlen, L4 ausreichend.\')\n    free, total = torch.cuda.mem_get_info(0)\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    disk = shutil.disk_usage(cache_dir).free\n    if free < 10 * (1 << 30):\n        raise RuntimeError(\'Mindestens 10 GiB freie GPU-VRAM für unverändertes FP32 erforderlich.\')\n    if disk < 8 * (1 << 30):\n        raise RuntimeError(\'Mindestens 8 GiB freier Cache-Speicher erforderlich; keine Gewichte heruntergeladen.\')\n    available = None\n    meminfo = Path(\'/proc/meminfo\')\n    if meminfo.exists():\n        fields = dict(line.split(\':\', 1) for line in meminfo.read_text().splitlines())\n        available = int(fields[\'MemAvailable\'].split()[0]) * 1024\n        if available < 10 * (1 << 30):\n            raise RuntimeError(\'Mindestens 10 GiB verfügbarer Host-RAM erforderlich.\')\n    return {\'gpu\': torch.cuda.get_device_name(0), \'free_vram\': free, \'total_vram\': total,\n            \'free_disk\': disk, \'available_host_ram\': available, \'device\': \'cuda:0\'}\n\n\ndef model_load_kwargs(torch):\n    return {\'revision\': REVISION, \'dtype\': torch.float32, \'device_map\': {\'\': 0},\n            \'attn_implementation\': \'sdpa\', \'trust_remote_code\': False, \'use_safetensors\': True,\n            \'local_files_only\': True}\n\n\ndef load_model(reference, cache_dir):\n    import torch\n    from huggingface_hub import snapshot_download\n    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer\n    snapshot = Path(snapshot_download(MODEL, revision=REVISION, cache_dir=str(cache_dir),\n                                     allow_patterns=[\'*.json\', \'*.safetensors\', \'*.jinja\']))\n    if snapshot.name != REVISION:\n        raise ValueError(\'HF snapshot path does not identify the exact pinned revision\')\n    for filename, expected in reference.model.sha256:\n        if sha_file(snapshot / filename) != expected:\n            raise ValueError(\'Original model/tokenizer SHA mismatch: \' + filename)\n    config = AutoConfig.from_pretrained(str(snapshot), revision=REVISION, trust_remote_code=False,\n                                       local_files_only=True)\n    if config.model_type != \'qwen3\' or config.num_hidden_layers != 28 or config.intermediate_size != 6144:\n        raise ValueError(\'Original Qwen3-1.7B architecture mismatch\')\n    tokenizer = AutoTokenizer.from_pretrained(str(snapshot), revision=REVISION, trust_remote_code=False,\n                                             local_files_only=True)\n    tokenizer.padding_side = \'left\'\n    model = AutoModelForCausalLM.from_pretrained(str(snapshot), config=config, **model_load_kwargs(torch))\n    model.eval().requires_grad_(False)\n    if any(parameter.device.type != \'cuda\' or parameter.device.index != 0 or parameter.dtype != torch.float32\n           for parameter in model.parameters()):\n        raise ValueError(\'All model parameters must remain full FP32 on cuda:0; no offload/quantization\')\n    return tokenizer, model\n\n\ndef prepare_request(tokenizer, index, item):\n    import torch\n    category, question, answer = item[\'category\'], item[\'prompt\'], item[\'target\']\n    formatted = tokenizer.apply_chat_template([{\'role\': \'user\', \'content\': question}], tokenize=False,\n                                              add_generation_prompt=True, enable_thinking=False)\n    prompt = tokenizer(formatted, add_special_tokens=False)[\'input_ids\']\n    answer_ids = tokenizer(answer, add_special_tokens=False)[\'input_ids\']\n    if not prompt or not answer_ids or len(prompt) + len(answer_ids) > 128:\n        raise ValueError(\'Unexpected empty or long fixed teacher-forcing request; no silent truncation\')\n    # Token at p predicts token p+1. The last answer activation is recorded but\n    # cannot cause its own prediction. Answer likelihood uses preceding positions.\n    ids = prompt + answer_ids\n    positions = list(range(max(0, len(ids) - 4), len(ids)))\n    return {\'request_id\': f\'r{index:03d}\', \'item_id\': item[\'id\'], \'pair_id\': item[\'pair_id\'], \'category\': category, \'question\': question, \'answer\': answer,\n            \'formatted_prompt\': formatted, \'formatted_prompt_sha256\': hashlib.sha256(formatted.encode()).hexdigest(), \'prompt_ids\': prompt, \'answer_ids\': answer_ids,\n            \'input_ids\': ids, \'capture_positions\': positions, \'pulse_positions\': [len(prompt) - 1],\n            \'answer_logit_positions\': list(range(len(prompt) - 1, len(ids) - 1)),\n            \'thinking_enabled\': False, \'ids\': torch.tensor([ids], dtype=torch.long)}\n\n\n@contextlib.contextmanager\ndef cett_tap(model, coordinates, positions, weight_norms=None):\n    import torch\n    groups = defaultdict(list)\n    for order, (layer, neuron) in enumerate(coordinates):\n        groups[layer].append((order, neuron))\n    result = {\'cett\': torch.empty((1, len(positions), len(coordinates)), dtype=torch.float32),\n              \'denominator\': torch.empty((1, len(positions), len(coordinates)), dtype=torch.float32)}\n    handles, seen = [], set()\n    try:\n        for layer, pairs in groups.items():\n            down = model.model.layers[layer].mlp.down_proj\n            columns = [neuron for _, neuron in pairs]\n            weight_norm = (weight_norms[layer] if weight_norms is not None\n                           else down.weight.float().norm(dim=0)[columns])\n\n            def hook(_module, args, output, layer=layer, pairs=pairs, columns=columns, weight_norm=weight_norm):\n                if layer in seen:\n                    raise ValueError(\'CETT tap fired repeatedly\')\n                z = args[0][:, positions][:, :, columns].abs().float()\n                den = output.norm(dim=-1)[:, positions].float()\n                contribution = z * weight_norm / den.clamp_min(1e-9)[..., None]\n                order = [index for index, _ in pairs]\n                result[\'cett\'][:, :, order] = contribution.detach().cpu()\n                result[\'denominator\'][:, :, order] = den[..., None].expand_as(contribution).detach().cpu()\n                seen.add(layer)\n            handles.append(down.register_forward_hook(hook))\n        yield result\n        if seen != set(groups):\n            raise ValueError(\'CETT capture incomplete\')\n    finally:\n        for handle in reversed(handles):\n            handle.remove()\n\n\ndef parameter_evidence(model):\n    return tuple((name, id(value), value.data_ptr(), value._version, str(value.dtype), str(value.device))\n                 for name, value in model.named_parameters())\n\n\ndef run_arm(model, request, coordinates, stats, arm, *, efficient_gpu=True, include_logits=False, weight_norms=None):\n    import torch\n\n    from attention_probe.interventions import AttentionPulse, CoordinateCapture\n    device = next(model.parameters()).device\n    ids = request[\'ids\'].to(device)\n    attention = (torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION)\n                 if efficient_gpu else contextlib.nullcontext())\n    started = time.monotonic()\n    with (torch.inference_mode(), attention,\n          CoordinateCapture(model, coordinates, request[\'capture_positions\'],\n                            residual_layers=tuple(layer for layer in (3, 7, 14, 27) if layer < len(model.model.layers)),\n                            expected_count=len(coordinates)) as capture,\n          cett_tap(model, coordinates, request[\'capture_positions\'], weight_norms) as contributions,\n          AttentionPulse(model, arm[\'block\'], request[\'pulse_positions\'], arm[\'beta\'], arm[\'mode\'],\n                         seed=SEED + int(request[\'request_id\'][1:])) as pulse):\n        output = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)\n        if not bool(torch.isfinite(output.logits).all()):\n            raise ValueError(\'Nonfinite model logits\')\n        selected_logits = output.logits[:, request[\'answer_logit_positions\']].float()\n        targets = torch.tensor(request[\'answer_ids\'], device=device)[None, :, None]\n        logprob = selected_logits.log_softmax(-1).gather(-1, targets).squeeze(-1).cpu()\n        raw = capture.values\n        cett = contributions[\'cett\']\n        mu = torch.tensor(stats[\'refw\'][\'c_mu\'], dtype=torch.float32)\n        sd = torch.tensor(stats[\'refw\'][\'c_sd\'], dtype=torch.float32).clamp_min(1e-6)\n        zcett = (cett - mu) / sd\n        result = {\'arm\': arm, \'answer_token_logprobs\': logprob.flatten().tolist(),\n                  \'answer_logprob_sum\': float(logprob.sum()), \'answer_logprob_mean\': float(logprob.mean()),\n                  \'pulse_fired\': pulse.fired_count, \'capture_fired\': capture.fired_count,\n                  \'requested_delta_norms\': pulse.requested_delta_norms.tolist(),\n                  \'actual_delta_norms\': pulse.actual_delta_norms.tolist(),\n                  \'elapsed_seconds\': time.monotonic() - started}\n        arrays = {\'raw_z\': raw.numpy(), \'cett\': cett.numpy(), \'zcett\': zcett.numpy(),\n                  \'down_output_norm_raw\': contributions[\'denominator\'].numpy(),\n                  \'gold_token_logprob\': logprob.numpy()}\n        for layer, value in capture.residuals.items():\n            arrays[f\'residual_layer_{layer}\'] = value.numpy()\n        if include_logits:\n            arrays[\'technical_logits\'] = selected_logits.detach().cpu().numpy()\n    return result, arrays\n\n\ndef plan_document(reference, stats, items):\n    import attention_probe.reference as reference_module\n    module_root = Path(reference_module.__file__).parent\n    module_hashes = {name: sha_file(module_root / name) for name in (\'reference.py\', \'interventions.py\', \'fixed_refw.py\')}\n    return {\'schema\': \'qwen3_h286_short_pilot_v1\', \'model\': reference.model.to_dict(),\n            \'hn_sha256\': reference.hn_sha256, \'policy_sha256\': reference.policy_sha256,\n            \'refw_sha256\': REFW_SHA256, \'refw_source_sha256\': stats[\'source_hn_meta_sha256\'],\n            \'coordinates\': [list(pair) for pair in reference.coordinates], \'seed\': SEED,\n            \'arms\': list(ARMS), \'items\': items, \'items_sha256\': ITEMS_SHA256,\n            \'helper_sha256\': sha_file(__file__), \'module_sha256\': module_hashes,\n            \'dtype\': \'float32\', \'tf32\': False, \'attention\': \'sdpa/EFFICIENT_ATTENTION\',\n            \'transformers\': TRANSFORMERS, \'thinking_enabled\': False,\n            \'claim\': \'externally supplied draft-checking proxy; not own prior output or proof of self-monitoring; no held-out confirmatory test\',\n            \'candidate_block\': \'14 zero-based, prespecified candidate; not a proven bottleneck\',\n            \'cett\': \'abs(z)*norm(Wdown_col)/max(norm(down(z)),1e-9)\',\n            \'zcett\': \'(CETT-fixed_REF_W_c_mu)/max(fixed_REF_W_c_sd,1e-6)\'}\n\n\ndef verified_request(output, request_id, plan_hash):\n    directory = output / \'requests\' / request_id\n    if not directory.exists():\n        return None\n    record = json.loads((directory / \'result.json\').read_text())\n    if (record.get(\'state\') != \'complete\' or record.get(\'plan_sha256\') != plan_hash\n            or record.get(\'request_id\') != request_id or len(record.get(\'arms\', [])) != 5\n            or [row[\'arm\'] for row in record[\'arms\']] != list(ARMS)):\n        raise ValueError(\'Existing request does not match fixed all-five-arm plan: \' + request_id)\n    proof = record.get(\'technical_checks\')\n    if not isinstance(proof, dict) or proof.get(\'state\') != \'passed\' or proof.get(\'plan_sha256\') != plan_hash:\n        raise ValueError(\'Request lacks bound technical-check evidence: \' + request_id)\n    if sha_file(directory / \'arrays.npz\') != record[\'npz_sha256\']:\n        raise ValueError(\'Existing request array SHA mismatch: \' + request_id)\n    return record\n\n\ndef restore_resume(resume_zip, output, plan_hash):\n    if not resume_zip:\n        return\n    with zipfile.ZipFile(resume_zip) as bundle:\n        source_plan = json.loads(bundle.read(\'run_plan.json\'))\n        if hashlib.sha256(canonical(source_plan)).hexdigest() != plan_hash:\n            raise ValueError(\'Resume ZIP belongs to a different immutable plan/reference\')\n        for index in range(16):\n            request_id = f\'r{index:03d}\'\n            prefix = f\'requests/{request_id}/\'\n            names = [prefix + \'result.json\', prefix + \'arrays.npz\']\n            present = [name in bundle.namelist() for name in names]\n            if not any(present):\n                continue\n            if not all(present):\n                raise ValueError(\'Resume ZIP contains an incomplete request chunk\')\n            destination = output / \'requests\' / request_id\n            if destination.exists():\n                verified_request(output, request_id, plan_hash)\n                if any((destination / Path(name).name).read_bytes() != bundle.read(name) for name in names):\n                    raise ValueError(\'Resume ZIP conflicts with existing complete request; no overwrite\')\n                continue\n            parent = destination.parent\n            parent.mkdir(parents=True, exist_ok=True)\n            temporary = Path(tempfile.mkdtemp(prefix=\'.\' + request_id + \'_\', dir=parent))\n            try:\n                for name in names:\n                    if bundle.getinfo(name).file_size > 128 * (1 << 20):\n                        raise ValueError(\'Oversized request chunk in resume ZIP\')\n                    (temporary / Path(name).name).write_bytes(bundle.read(name))\n                record = json.loads((temporary / \'result.json\').read_text())\n                if record.get(\'plan_sha256\') != plan_hash or sha_file(temporary / \'arrays.npz\') != record.get(\'npz_sha256\'):\n                    raise ValueError(\'Resume request failed hash/plan verification\')\n                os.rename(temporary, destination)\n                verified_request(output, request_id, plan_hash)\n            finally:\n                if temporary.exists():\n                    shutil.rmtree(temporary)\n\n\ndef write_request(output, request, results, arrays, plan_hash, hardware, technical_checks=None):\n    import numpy as np\n    destination = output / \'requests\' / request[\'request_id\']\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.exists():\n        raise ValueError(\'Refusing to overwrite existing request outputs\')\n    temporary = Path(tempfile.mkdtemp(dir=destination.parent, prefix=\'.\' + destination.name + \'_\'))\n    try:\n        np.savez_compressed(temporary / \'arrays.npz\',\n                            **{key: np.stack([row[key] for row in arrays]) for key in arrays[0]})\n        with (temporary / \'arrays.npz\').open(\'rb\') as stream:\n            os.fsync(stream.fileno())\n        public_request = {key: value for key, value in request.items() if key != \'ids\'}\n        record = {\'state\': \'complete\', \'request_id\': request[\'request_id\'], \'request\': public_request,\n                  \'plan_sha256\': plan_hash, \'npz_sha256\': sha_file(temporary / \'arrays.npz\'), \'arms\': results,\n                  \'hardware\': hardware, \'technical_checks\': technical_checks, \'completed_utc\': dt.datetime.now(dt.UTC).isoformat()}\n        atomic_json(temporary / \'result.json\', record)\n        os.rename(temporary, destination)  # Both files become visible as one paired-request checkpoint.\n        return record\n    finally:\n        if temporary.exists():\n            shutil.rmtree(temporary)\n\n\ndef run(hn_set, policy, refw, items_path, out_dir, cache_dir=None, resume_zip=None):\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    status = {\'state\': \'running\', \'stage\': \'reference\', \'completed_requests\': 0,\n              \'confirmatory_eligible\': False, \'scope\': \'original_fixed_H286_exploratory_pilot\'}\n    metrics = {\'pretrained_model_run\': False, \'checks\': {}, \'no_training\': True}\n    started = time.monotonic()\n    def save():\n        atomic_json(output / \'status.json\', status)\n        atomic_json(output / \'metrics.json\', metrics)\n    save()\n    try:\n        reference, stats, items = load_inputs(hn_set, policy, refw, items_path)\n        plan = plan_document(reference, stats, items)\n        plan_hash = hashlib.sha256(canonical(plan)).hexdigest()\n        existing = output / \'run_plan.json\'\n        if existing.exists() and existing.read_bytes() != canonical(plan) + b\'\\n\':\n            raise ValueError(\'Existing output uses a different plan; choose a fresh directory\')\n        if not existing.exists():\n            atomic_json(existing, plan)\n        restore_resume(resume_zip, output, plan_hash)\n        done = [verified_request(output, f\'r{index:03d}\', plan_hash) for index in range(len(items))]\n        status[\'completed_requests\'] = sum(item is not None for item in done)\n        if all(item is not None for item in done):\n            metrics[\'technical_checks_from_completed_requests\'] = [row.get(\'technical_checks\') for row in done]\n            metrics[\'checks_executed_this_attempt\'] = False\n            if any(not isinstance(row.get(\'technical_checks\'), dict) or row[\'technical_checks\'].get(\'state\') != \'passed\'\n                   or row[\'technical_checks\'].get(\'plan_sha256\') != plan_hash for row in done):\n                raise ValueError(\'Completed resume lacks the bound technical-check evidence\')\n            status.update(state=\'complete\', stage=\'complete\', resumed_without_gpu=True)\n            return status\n        from importlib.metadata import version\n\n        import torch\n        import transformers\n        if transformers.__version__ != TRANSFORMERS or version(\'accelerate\') != ACCELERATE:\n            raise ValueError(\'Exact transformers5.22.2 and accelerate1.16.0 are required\')\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.set_float32_matmul_precision(\'highest\')\n        torch.manual_seed(SEED)\n        disable_bar = getattr(transformers.utils.logging, \'disable_progress_bar\', None)\n        if disable_bar is not None:\n            disable_bar()\n        metrics.update(torch=torch.__version__, transformers=transformers.__version__, python=platform.python_version())\n        status[\'stage\'] = \'hardware_preflight\'\n        cache = Path(cache_dir or Path.home() / \'.cache\' / \'h286_pilot\')\n        hardware = hardware_preflight(torch, cache)\n        metrics[\'hardware\'] = hardware\n        status[\'stage\'] = \'pinned_download_and_load\'\n        save()\n        os.environ[\'HF_DEACTIVATE_ASYNC_LOAD\'] = \'1\'\n        print(\'Status: Original Qwen3-1.7B laden, exakte Dateien prüfen; FP32/SDPA.\', flush=True)\n        tokenizer, model = load_model(reference, cache)\n        metrics[\'weights_loaded\'] = True\n        hardware.update(torch=torch.__version__, cuda=torch.version.cuda, python=platform.python_version(), compute_capability=list(torch.cuda.get_device_capability(0)), transformers=transformers.__version__, attention=\'sdpa/EFFICIENT_ATTENTION\', dtype=\'float32\', tf32=False)\n        groups = defaultdict(list)\n        for layer, neuron in reference.coordinates:\n            groups[layer].append(neuron)\n        with torch.inference_mode():\n            weight_norms = {layer: model.model.layers[layer].mlp.down_proj.weight.float().norm(dim=0)[columns]\n                            for layer, columns in groups.items()}\n        evidence = parameter_evidence(model)\n        first_index = next(index for index, result in enumerate(done) if result is None)\n        warm = prepare_request(tokenizer, first_index, items[first_index])\n        ids = warm[\'ids\'].to(\'cuda:0\')\n        status[\'stage\'] = \'warmup\'\n        save()\n        with torch.inference_mode(), torch.nn.attention.sdpa_kernel(torch.nn.attention.SDPBackend.EFFICIENT_ATTENTION):\n            model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)  # untimed warmup\n            baseline = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits.detach().cpu()\n        metrics[\'pretrained_model_run\'] = True\n        status[\'stage\'] = \'technical_checks\'\n        print(\'Status: Zwei technische Prüfungen vor der festen 16-Request-Auswertung.\', flush=True)\n        _sham, sham_arrays = run_arm(model, warm, reference.coordinates, stats, ARMS[0], include_logits=True, weight_norms=weight_norms)\n        selected_baseline = baseline[:, warm[\'answer_logit_positions\']].float()\n        observed = torch.from_numpy(sham_arrays[\'technical_logits\'])\n        delta = (selected_baseline - observed).abs()\n        metrics[\'checks\'][\'sham\'] = {\'bitwise_equal\': torch.equal(selected_baseline, observed),\n                                     \'max_abs_logit_difference\': float(delta.max()), \'rtol\': 1e-5, \'atol\': 1e-6}\n        if not torch.allclose(selected_baseline, observed, rtol=1e-5, atol=1e-6):\n            raise ValueError(\'Technical sham logit check failed; exploratory queue not started\')\n        pulse_result, pulse_arrays = run_arm(model, warm, reference.coordinates, stats, ARMS[1], weight_norms=weight_norms)\n        import numpy as np\n        upstream = [index for index, (layer, _) in enumerate(reference.coordinates) if layer < 14]\n        earlier = [index for index, position in enumerate(warm[\'capture_positions\']) if position < warm[\'pulse_positions\'][0]]\n        upstream_equal = np.array_equal(sham_arrays[\'raw_z\'][..., upstream], pulse_arrays[\'raw_z\'][..., upstream])\n        earlier_equal = np.array_equal(sham_arrays[\'raw_z\'][:, earlier], pulse_arrays[\'raw_z\'][:, earlier])\n        metrics[\'checks\'][\'causality\'] = {\'upstream_coordinates\': len(upstream), \'upstream_raw_equal\': upstream_equal,\n                                           \'earlier_positions\': len(earlier), \'earlier_raw_equal\': earlier_equal}\n        if not upstream_equal or not earlier_equal:\n            raise ValueError(\'Technical causal guard failed; exploratory queue not started\')\n        metrics[\'checks\'][\'pulse_and_capture\'] = {\'pulse_once\': pulse_result[\'pulse_fired\'] == 1,\n                                                  \'capture_once\': pulse_result[\'capture_fired\'] == 1,\n                                                  \'frozen_parameter_evidence\': parameter_evidence(model) == evidence}\n        if parameter_evidence(model) != evidence:\n            raise ValueError(\'Model parameter identity/storage/version counters changed\')\n        metrics[\'checks_executed_this_attempt\'] = True\n        technical_checks = {\'state\': \'passed\', \'plan_sha256\': plan_hash, \'checks\': metrics[\'checks\'],\n                            \'hardware\': hardware, \'executed_utc\': dt.datetime.now(dt.UTC).isoformat()}\n        measured = []\n        for index, item in enumerate(items):\n            if done[index] is not None:\n                continue\n            request = prepare_request(tokenizer, index, item)\n            status.update(stage=\'exploratory_requests\', current_request=request[\'request_id\'])\n            save()\n            print(f"Request {index + 1}/16: {request[\'request_id\']} — fünf feste Arme", flush=True)\n            t0 = time.monotonic()\n            order = list(range(len(ARMS)))\n            random.Random(SEED + index).shuffle(order)\n            request[\'execution_order\'] = [ARMS[arm_index][\'name\'] for arm_index in order]\n            results, array_rows = [None] * len(ARMS), [None] * len(ARMS)\n            cpu_rng, cuda_rng = torch.random.get_rng_state().clone(), torch.cuda.get_rng_state().clone()\n            for arm_index in order:\n                result, arrays = run_arm(model, request, reference.coordinates, stats, ARMS[arm_index], weight_norms=weight_norms)\n                results[arm_index], array_rows[arm_index] = result, arrays\n            if not torch.equal(cpu_rng, torch.random.get_rng_state()) or not torch.equal(cuda_rng, torch.cuda.get_rng_state()):\n                raise ValueError(\'Global RNG changed during fixed five-arm request\')\n            if parameter_evidence(model) != evidence:\n                raise ValueError(\'Frozen parameter evidence changed; request is not published complete\')\n            record = write_request(output, request, results, array_rows, plan_hash, hardware, technical_checks)\n            done[index] = record\n            measured.append(time.monotonic() - t0)\n            status[\'completed_requests\'] = sum(row is not None for row in done)\n            remaining = len(items) - status[\'completed_requests\']\n            eta_seconds = sum(measured) / len(measured) * remaining\n            finish = dt.datetime.now(ZoneInfo(\'Europe/Berlin\')) + dt.timedelta(seconds=eta_seconds)\n            status.update(estimated_remaining_seconds=eta_seconds, estimated_finish_berlin=finish.isoformat())\n            save()\n            print(f"Gespeichert {status[\'completed_requests\']}/16; Restschätzung {eta_seconds / 60:.1f} min; Berlin {finish:%H:%M}", flush=True)\n            if remaining and (eta_seconds > 3600 or finish.date() > dt.datetime.now(ZoneInfo(\'Europe/Berlin\')).date()):\n                status.update(state=\'paused\', stage=\'safe_request_checkpoint\',\n                              reason=\'Runtime estimate exceeds one hour or today; resume ZIP keeps paired requests together\')\n                break\n        else:\n            status.update(state=\'complete\', stage=\'complete\')\n        metrics[\'weights_unchanged_evidence\'] = \'same parameter identities, storage pointers, version counters; no training; no full tensor rehash\'\n        metrics[\'hardware_comparison_warning\'] = \'Each complete paired request stays on one GPU; resumed chunks can differ by GPU/software.\'\n    except BaseException as exc:  # noqa: BLE001 — preserve diagnostics also on interruption\n        status.update(state=\'failed\', error_type=type(exc).__name__, error=str(exc))\n        (output / \'traceback.txt\').write_text(traceback.format_exc())\n        print(\'HARD FAILURE: \' + str(exc), flush=True)\n    finally:\n        metrics[\'elapsed_seconds\'] = time.monotonic() - started\n        save()\n        archive(output)\n    return status\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--hn-set\', required=True)\n    parser.add_argument(\'--policy\', required=True)\n    parser.add_argument(\'--refw\', required=True)\n    parser.add_argument(\'--items\', required=True)\n    parser.add_argument(\'--out-dir\', required=True)\n    parser.add_argument(\'--cache-dir\')\n    parser.add_argument(\'--resume-zip\')\n    args = parser.parse_args()\n    result = run(args.hn_set, args.policy, args.refw, args.items, args.out_dir, args.cache_dir, args.resume_zip)\n    print(json.dumps(result), flush=True)\n    raise SystemExit(0 if result[\'state\'] in (\'complete\', \'paused\') else 1)\n'}
MODULE_SHA256 = {'attention_probe/__init__.py': 'de626db34130b6d517d0dc776df8eff3b794bec5d7c9ba70157abe016f422705', 'attention_probe/reference.py': 'e89bd465d0e962e840ba589280ae89b84dd74f6fda1a459f4c76826ef4bd51e3', 'attention_probe/fixed_refw.py': 'b5cb362889f473f8b902454b4e22f4a615a54ed0a2bac0d7b0cc4c80e3c778e3', 'attention_probe/interventions.py': '1c7afb046108e2d7a214e1849198574128acf2c3e300888cdb6cc47e0e086561', 'attention_probe/pilot_items.json': '12a3cba862185b9f25ca39ea3ea49bd3bb6fc5f248c12dbbe02bcc8836663fb3', 'qwen3_h286_runtime.py': 'e30b9da80bd9370eb5de04245ad43b8c0ac0e292611ba32d1a8721b28e1e4312'}


In [ ]:
import base64
from collections import deque
import datetime
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import subprocess
import sys
import traceback
import zipfile

base = Path('/content') if Path('/content').is_dir() else Path.cwd()
binding_fingerprint = hashlib.sha256(json.dumps({'modules': MODULE_SHA256, 'references': REFERENCE_SHA256},
                                                sort_keys=True).encode()).hexdigest()[:12]
work = (Path(RUN_DIRECTORY).expanduser() if RUN_DIRECTORY
        else base / ('h286_attention_pilot_' + binding_fingerprint))
output = work / 'output'
output.mkdir(parents=True, exist_ok=True)
failure = None
helper_started = False
helper_status = None
return_code = None
status = {'state': 'running', 'stage': 'gpu_preflight', 'purpose': 'original_qwen3_h286_attention_pilot',
          'pretrained_weights_loaded': False, 'inference': 'exploratory (post hoc)'}

def write_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + chr(10))

def run_command(command, logname, env=None):
    process = None
    tail = deque(maxlen=10)
    shown = 0
    try:
        with (output / logname).open('a', encoding='utf-8') as log:
            process = subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                log.write(line); log.flush()
                short = line.strip().replace(chr(13), ' ')[:500]
                tail.append(short)
                if logname == 'runtime.log' and shown < 120 and short.startswith(
                        ('Request ', 'Gespeichert ', 'Stage ', 'Status:', 'Status ', 'CHECK ', 'HARD FAILURE:')):
                    print(short, flush=True); shown += 1
            code = process.wait()
        if code:
            print('Kindprozess fehlgeschlagen; vollständiges Log im ZIP. Letzte Zeilen:', flush=True)
            for short in tail:
                print(short, flush=True)
            raise subprocess.CalledProcessError(code, command)
        return code
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
        raise

try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Keine CUDA-GPU. A100, L4 oder B200 auswählen; kein Warten.')
    capability = tuple(torch.cuda.get_device_capability(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    if capability[0] < 8 or free_bytes < MIN_FREE_VRAM_GIB * 1024**3:
        raise RuntimeError('Ampere-oder-neuere CUDA-GPU mit mindestens 12 GiB freiem Speicher erforderlich.')
    torch_before = importlib.metadata.version('torch')
    environment = {'python': sys.version, 'torch': torch_before, 'torch_cuda': torch.version.cuda,
                   'gpu': torch.cuda.get_device_name(0), 'capability': list(capability),
                   'free_vram_gib': free_bytes / 1024**3, 'total_vram_gib': total_bytes / 1024**3,
                   'transformers_required': TRANSFORMERS_VERSION, 'accelerate_required': ACCELERATE_VERSION,
                   'execution_dtype': 'float32', 'tf32': False}
    write_json(output / 'environment_before.json', environment)
    print('GPU:', environment['gpu'], '| frei:', round(environment['free_vram_gib'], 2), 'GiB')
    print('Torch/CUDA bleibt erhalten:', torch_before, torch.version.cuda)
    status['stage'] = 'reference_validation'
    inputs = work / 'inputs'; inputs.mkdir(exist_ok=True)
    for name, expected in REFERENCE_SHA256.items():
        if name in REFERENCE_FILES_B64:
            raw = base64.b64decode(REFERENCE_FILES_B64[name], validate=True)
        elif REFERENCE_DIRECTORY:
            raw = (Path(REFERENCE_DIRECTORY).expanduser() / name).read_bytes()
        else:
            raise RuntimeError('Öffentliches Template ohne private Referenz. Die bereitgestellte private Nutzdatei verwenden.')
        if hashlib.sha256(raw).hexdigest() != expected:
            raise ValueError('Originalreferenz-Prüfsumme weicht ab: ' + name)
        target = inputs / name
        if target.exists() and target.read_bytes() != raw:
            raise ValueError('Vorhandene Referenz gehört zu einer anderen Auswahl; Ergebnisse nicht mischen.')
        target.write_bytes(raw)
    binding = {'sha256': MODULE_SHA256, 'reference_sha256': REFERENCE_SHA256,
               'model': 'Qwen/Qwen3-1.7B', 'revision': '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'}
    binding_path = output / 'embedded_source.json'
    if binding_path.exists() and json.loads(binding_path.read_text()) != binding:
        raise ValueError('Vorhandener Lauf gehört zu anderem Quellcode oder Referenzen; Ergebnisse nicht mischen.')
    code = work / 'code'; code.mkdir(exist_ok=True)
    for name, source in MODULE_SOURCES.items():
        if hashlib.sha256(source.encode()).hexdigest() != MODULE_SHA256[name]:
            raise ValueError('Eingebetteter Quellcode weicht ab: ' + name)
        path = code / name; path.parent.mkdir(parents=True, exist_ok=True); path.write_text(source)
    write_json(binding_path, binding)
    status['stage'] = 'dependencies'
    constraint = work / 'preserve_torch.txt'
    constraint.write_text('torch==' + torch_before + chr(10))
    print('Exakte Transformers-/Accelerate-Version installieren; GPU-Torch bleibt festgelegt.', flush=True)
    child_env = dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS='1', TQDM_DISABLE='1',
                     TOKENIZERS_PARALLELISM='false')
    run_command([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--upgrade-strategy', 'only-if-needed',
                 '--constraint', str(constraint), 'transformers==' + TRANSFORMERS_VERSION,
                 'accelerate==' + ACCELERATE_VERSION], 'install.log', child_env)
    installed = {name: importlib.metadata.version(name) for name in ('torch', 'transformers', 'accelerate')}
    if installed != {'torch': torch_before, 'transformers': TRANSFORMERS_VERSION, 'accelerate': ACCELERATE_VERSION}:
        raise RuntimeError('Paketversionen weichen ab; keine Modellgewichte angefordert.')
    write_json(output / 'environment_installed.json', installed)
    status['stage'] = 'runtime'
    helper_started = True
    status['pretrained_weights_loaded'] = None
    env = dict(child_env, PYTHONPATH=str(code))
    command = [sys.executable, str(code / 'qwen3_h286_runtime.py'), '--hn-set', str(inputs / 'hn_set.json'),
               '--policy', str(inputs / 'htq_policy.json'), '--refw', str(inputs / 'refw_h286.json'),
               '--items', str(code / 'attention_probe/pilot_items.json'),
               '--out-dir', str(output), '--cache-dir', str(base / 'h286_model_cache')]
    if RESUME_ZIP:
        command += ['--resume-zip', str(Path(RESUME_ZIP).expanduser())]
    print('Frischer Python-Prozess: Original-FP32-Modell, feste H286/REF-W, 16 kurze Aufgaben.', flush=True)
    return_code = run_command(command, 'runtime.log', env)
    helper_status = json.loads((output / 'status.json').read_text())
    if helper_status.get('state') not in ('complete', 'paused'):
        raise RuntimeError('Runtime hat keinen vollständigen oder bewusst pausierten Abschluss gemeldet.')
    status.update(helper_status, helper_return_code=return_code, wrapper_verified=True)
except BaseException as error:
    failure = error
    status.update(state='failed', error_type=type(error).__name__, error=str(error), wrapper_verified=False)
    (output / 'wrapper_diagnostic.txt').write_text(traceback.format_exc())
finally:
    # Keep the child's original status for diagnosis, then publish one authoritative final status.
    child_path = output / 'status.json'
    if helper_started and child_path.exists():
        try:
            helper_status = json.loads(child_path.read_text())
            write_json(output / 'helper_status.json', helper_status)
            if failure is not None:
                status['helper_state'] = helper_status.get('state')
                status['pretrained_weights_loaded'] = helper_status.get('pretrained_weights_loaded')
        except Exception:
            status['helper_state'] = 'unreadable'
    status['written_utc'] = datetime.datetime.now(datetime.timezone.utc).isoformat()
    write_json(output / 'status.json', status)
    members = {str(path.relative_to(output)): hashlib.sha256(path.read_bytes()).hexdigest()
               for path in sorted(output.rglob('*')) if path.is_file()
               and path.name not in ('WRAPPER_MANIFEST.json',) and path.suffix not in ('.zip', '.tmp')
               and not any(part.startswith('.') for part in path.relative_to(output).parts)}
    write_json(output / 'WRAPPER_MANIFEST.json', members)
    target = base / ('h286_l4_pilot_diagnostic.zip' if failure is not None else 'h286_l4_pilot_results.zip')
    temporary = target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(output.rglob('*')):
            if (path.is_file() and path.suffix not in ('.zip', '.tmp')
                    and not any(part.startswith('.') for part in path.relative_to(output).parts)):
                archive.write(path, str(path.relative_to(output)))
    with zipfile.ZipFile(temporary) as archive:
        if archive.testzip() is not None:
            raise RuntimeError('Ergebnis-ZIP ist beschädigt.')
    os.replace(temporary, target)
    print('Status:', status['state'], '| ZIP:', target, flush=True)
    try:
        from google.colab import files
        files.download(str(target))
    except Exception:
        try:
            from IPython.display import FileLink, display
            display(FileLink(os.path.relpath(target, Path.cwd())))
        except Exception:
            print('ZIP aus diesem Verzeichnis herunterladen:', target)
if failure is not None:
    raise RuntimeError('Versuch angehalten; das Diagnose-ZIP enthält den tatsächlichen Fehler.') from None
if status['state'] == 'paused':
    print('Bewusst pausierter Checkpoint; kein abgeschlossener Versuch und kein PASS.')
else:
    print('Begrenzter explorativer H286-Versuch abgeschlossen. Keine allgemeine Wahrheitsvalidierung.')


Das ZIP enthält Status, Messwerte, Zeitangaben und fortsetzbare Einzelaufgaben. GPU danach bei Bedarf beenden.
